In [5]:
import re
import pandas as pd
import numpy as np
from scipy import stats
from google.colab import files

In [8]:
up = files.upload()
nama_file = list(up.keys())[0]
print("File terbaca:", nama_file)

Saving Blank Quiz (Responses 3 TIC).xlsx to Blank Quiz (Responses 3 TIC) (1).xlsx
File terbaca: Blank Quiz (Responses 3 TIC) (1).xlsx


In [9]:
df = pd.read_excel(nama_file)
butir = df[[c for c in df.columns if "[" in c]]
print("Jumlah responden :", len(df))
print("Jumlah butir     :", butir.shape[1])
print("Data kosong      :", int(butir.isna().sum().sum()))

Jumlah responden : 40
Jumlah butir     : 29
Data kosong      : 0


In [10]:
skala = {
    "Sangat Tidak Setuju (STS)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5,
}
num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")
print("Sel yang gagal dikonversi:", int(num.isna().sum().sum()))

Sel yang gagal dikonversi: 51


/tmp/ipykernel_4357/906843197.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")


In [11]:
dimensi = {}
for kolom in butir.columns:
    nama = re.match(r"^(.*?)\s*\[", kolom).group(1).strip()
    dimensi.setdefault(nama, []).append(kolom)

for nama, daftar in dimensi.items():
    print(f"{nama}: {len(daftar)} butir")

Penggunaan AI pada tugas atau projek kuliah: 7 butir
Aspek Psikologis penggunaan AI pada tugas atau projek kuliah: 7 butir
Kebiasaan penggunaan teknologi digital: 6 butir
Prokrastinasi akademik.: 4 butir
Respon emosional: 5 butir


In [12]:
n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt((n - 2) + t**2)
print(f"n = {n} | r tabel (α=0,05; df={n-2}) = {r_tabel:.3f}")

hasil_validitas = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    total = x.sum(axis=1)
    for i, kolom in enumerate(daftar, 1):
        r, p = stats.pearsonr(x[kolom], total)
        r_koreksi, _ = stats.pearsonr(x[kolom], total - x[kolom])
        hasil_validitas.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": round(r, 3),
            "Corrected r": round(r_koreksi, 3),
            "p-value": round(p, 4),
            "Keterangan": "Valid" if r > r_tabel else "Tidak valid",
        })

tabel_validitas = pd.DataFrame(hasil_validitas)
tabel_validitas

n = 40 | r tabel (α=0,05; df=38) = 0.312


,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.413,0.217,0.0082,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.365,0.199,0.0206,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.635,0.432,0.0000,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,NaN,NaN,NaN,Tidak valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.774,0.568,0.0000,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.622,0.411,0.0000,Valid
6,Penggunaan AI pada tugas atau projek kuliah,B7,NaN,NaN,NaN,Tidak valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.550,0.419,0.0002,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,NaN,NaN,NaN,Tidak valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,NaN,NaN,NaN,Tidak valid


In [13]:
def cronbach_alpha(x):
    k = x.shape[1]
    return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori(a):
    return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80
            else "Reliabel" if a >= 0.70 else "Tidak reliabel")

hasil_reliabilitas = []
alpha_dihapus = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    a = cronbach_alpha(x)
    hasil_reliabilitas.append({"Dimensi": nama, "Jumlah butir": len(daftar),
                               "Cronbach's alpha": round(a, 3), "Kategori": kategori(a)})
    for i, kolom in enumerate(daftar, 1):
        if len(daftar) > 2:
            alpha_dihapus.append({"Dimensi": nama, "Butir": f"B{i}",
                                  "Alpha jika dihapus": round(cronbach_alpha(x.drop(columns=kolom)), 3)})

tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)
display(tabel_reliabilitas)
display(tabel_alpha_dihapus)

,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,7,0.619,Tidak reliabel
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.831,Baik
2,Kebiasaan penggunaan teknologi digital,6,0.917,Sangat baik
3,Prokrastinasi akademik.,4,1.032,Sangat baik
4,Respon emosional,5,0.752,Reliabel


,Dimensi,Butir,Alpha jika dihapus
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.614
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.617
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.547
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.593
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.479
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.554
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.679
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.830
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.805
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.704


In [14]:
print("Valid menurut r hitung       :", (tabel_validitas["r hitung"] > r_tabel).sum())
print("Valid menurut corrected r>=.30:", (tabel_validitas["Corrected r"] >= 0.30).sum())

Valid menurut r hitung       : 9
Valid menurut corrected r>=.30: 6


In [16]:
gabung = tabel_validitas.merge(tabel_alpha_dihapus, on=["Dimensi", "Butir"], how="left")
with pd.ExcelWriter("Hasil_Awal.xlsx") as w:
    gabung.to_excel(w, sheet_name="Validitas", index=False)
    tabel_reliabilitas.to_excel(w, sheet_name="Reliabilitas", index=False)
files.download("Hasil_Awal.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
daftar_aspek = dimensi["Aspek Psikologis penggunaan AI pada tugas atau projek kuliah"]
buang = [daftar_aspek[5]]    # Aspek Psikologis B6

num = num.drop(columns=buang)
for nama in dimensi:
    dimensi[nama] = [k for k in dimensi[nama] if k not in buang]
print("Butir tersisa:", num.shape[1])

Butir tersisa: 28


In [18]:
gabung = tabel_validitas.merge(tabel_alpha_dihapus, on=["Dimensi", "Butir"], how="left")
with pd.ExcelWriter("hasil_revisi.xlsx") as w:
    gabung.to_excel(w, sheet_name="Validitas", index=False)
    tabel_reliabilitas.to_excel(w, sheet_name="Reliabilitas", index=False)
files.download("hasil_revisi.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Latihan 4**

In [19]:
num_awal = butir.replace(skala).apply(pd.to_numeric, errors="coerce")
dim_awal = {}
for kolom in butir.columns:
    nama = re.match(r"^(.*?)\s*\[", kolom).group(1).strip()
    dim_awal.setdefault(nama, []).append(kolom)

sub = num_awal.sample(20, random_state=1)
n20 = len(sub)
t20 = stats.t.ppf(0.975, n20 - 2)
r_tabel20 = t20 / np.sqrt((n20 - 2) + t20**2)
print(f"n = {n20} | r tabel = {r_tabel20:.3f}")

jml_valid = 0
for nama, daftar in dim_awal.items():
    x = sub[daftar]
    total = x.sum(axis=1)
    for kolom in daftar:
        r, _ = stats.pearsonr(x[kolom], total)
        jml_valid += r > r_tabel20
    print(f"{nama[:30]:30s} alpha = {cronbach_alpha(x):.3f}")
print("Jumlah butir valid:", jml_valid, "dari", num_awal.shape[1])

n = 20 | r tabel = 0.444
Penggunaan AI pada tugas atau  alpha = 0.665
Aspek Psikologis penggunaan AI alpha = 0.815
Kebiasaan penggunaan teknologi alpha = 0.959
Prokrastinasi akademik.        alpha = 1.074
Respon emosional               alpha = 0.825
Jumlah butir valid: 9 dari 29


/tmp/ipykernel_4357/3058489433.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  num_awal = butir.replace(skala).apply(pd.to_numeric, errors="coerce")
